In [1]:
# Importing required Python Libraries
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from IPython.display import Markdown, display
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [2]:
# Loading required variables from environment
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
OPENAI_API_KEY = openai_key
SENDGRID_KEY = os.environ.get('SENDGRID_API_KEY')
print(SENDGRID_KEY[:6])

SG.CpD


In [3]:

product_manager_instructions = """
You are Product Manager for the Charity organization called Sollas Gardens. You are required to produce a product requirement document for developing a charity website. Your product requirement document should capture below key requirements,
     1. User registration portal,
     2. Validate the user with their ID
     3. Their authentication must be maintained via the portal
     4. Portal should have Personal dashboard explaining about upcoming events
     5. Selecting on each events should take to event registration page
     6. Then there should be a payment gateway integration for the event registration
     7. Upon successful registration user should get email address about their registration and event details.
    """

architect_instructions = """
You are Senior Architect for the Charity organization called Sollas Gardens. You are required to produce a architectural instruction for developing a charity website. You have a monthly budget not more than  100 USD and you must design a website platform that can support up to 1000 users. You must also provide the top 3 best practices for the website architecture and how to implement them. Your architectural instruction should capture below key requirements,
     1. User registration portal,
     2. Validate the user with their ID
     3. Their authentication must be maintained via the portal
     4. Portal should have Personal dashboard explaining about upcoming events
     5. Selecting on each events should take to event registration page
     6. Then there should be a payment gateway integration for the event registration
     7. Upon successful registration user should get email address about their registration and event details.
     You must create three separate suggestions to deploy this in three major cloud providers such as AWS, Azure and GCP with the cost breakdown for each of the services you are suggesting.
     Your framework must suggest a Python based backend framework and frontend UI framework which be used with minimal coding efforts using AI
     """

developer_instruction = """
You are part of the approval board which is expected to review the product managers requirements and architect suggestion and then look at the budget you can offer within USD 100.  You must be clear and robust in giving the feedback where additional cost can be saved while designing this
"""

sre_instructions = """You will need to look into the Product, Architect and Engineer requirements and then provide suggestion to how to effectively monitor the system and maintain the SLO."""

In [4]:
# Defining an agent

product_agent = Agent(
    name="Product Manager",
    instructions=product_manager_instructions,
    model="gpt-4o-mini"
)

architect_agent = Agent(
    name="Architect",
    instructions=architect_instructions,
    model="gpt-4o-mini"
)

developer_agent = Agent(
    name="Approval Board",
    instructions=developer_instruction,
    model="gpt-4o-mini"
)

site_reliability_agent = Agent(
    name="Site Reliability Engineer",
    instructions=sre_instructions,
    model="gpt-4o-mini"
)

In [5]:
# invoking an agent with Trace (trace available at https://platform.openai.com/logs?api=traces)
@function_tool
def send_email(body: str):
        sg = SendGridAPIClient(SENDGRID_KEY)
        from_email=Email('ajay291491@gmail.com')
        to_email = To("ajay291491@gmail.com")
        subject="Charity Website Project Report"
        content=Content("text/html", body)
        mail = Mail(from_email, to_email, subject, content).get()
        try:
            sg.send(mail)
        except Exception as e:
            print(f"Error sending email: {str(e)}")

In [6]:
description = """Create a complete project report by combing all suggestions from Product, architect, developer and site reliability engineer"""

product_tool = product_agent.as_tool(tool_name="product_agent",tool_description="Produce a product requirement document for developing a charity website")
architect_tool = architect_agent.as_tool(tool_name="architect_agent",tool_description="Produce a architectural instruction for developing a charity website")
developer_tool = developer_agent.as_tool(tool_name="developer_agent",tool_description="Produce a Developer  instruction for developing a charity website")
sre_tool = site_reliability_agent.as_tool(tool_name="sre_agent",tool_description="Produce a SRE instruction for developing a charity website")

project_tools = [product_tool, architect_tool, developer_tool, sre_tool, send_email]
project_tools

[FunctionTool(name='product_agent', description='Produce a product requirement document for developing a charity website', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7c1d629bb140>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False),
 FunctionTool(name='architect_agent', description='Produce a architectural instruction for developing a charity website', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'Agent

In [ ]:
instructions = """You are a project manager who is responsible to create a project report for the charity website development project. You have access to 4 different agents who are experts in their respective domains such as product management, architecture, development and site reliability. You can call these agents as tools to get their suggestions and then combine all the suggestions to create a detailed project report as given by agents. Once the report is ready you must convert the report in HTML content and then send this report via email to the stakeholders. You must use the send_email tool to send the email. The subject of the email should be 'Charity Website Development Project Report'"""

project_agent = Agent(
    name="Project Manager",
    instructions=instructions,
    model="gpt-4o-mini",
    tools=project_tools
)
message = "send project report to Stakeholders"

with trace("Charity Website Project Report Generation"):
    result = await Runner.run(project_agent, message)
    output = result.final_output
    display(Markdown(output))